# Encouraging parallel tool calls with a batch tool

By default, Claude can call several tools in a single assistant turn (parallel tool use is on unless you set `disable_parallel_tool_use`), and current models usually do so on their own when a request clearly needs more than one tool. A model may still choose to call tools one at a time on some prompts, which costs an extra round trip per call. One way to make batching explicit is a "batch tool": a meta-tool whose only job is to wrap several invocations of your other tools, which your code then unpacks and executes together.

This notebook first shows the plain multi-tool case and a loop that answers every `tool_use` block in a turn, then adds the batch tool so you can compare. Treat the batch tool as an optional pattern: if the model already returns all the calls you need in one turn (as it does in the recorded run below), you don't need it.

In [1]:
from anthropic import Anthropic

client = Anthropic()
MODEL_NAME = "claude-sonnet-5-5"

## Performing a query with multiple tool calls

Recall that the default behavior is for Claude to be allowed parallel tool calls. Combined with the default `tool_choice` of `auto`, this means that Claude can call any of the specified tools, or call more than one of them in a single assistant turn.

Let's set Claude up with a `get_weather` and `get_time` tool.

In [2]:
def get_weather(location):
    # Pretend to get the weather, and just return a fixed value.
    return f"The weather in {location} is 72 degrees and sunny."


def get_time(location):
    # Pretend to get the time, and just return a fixed value.
    return f"The time in {location} is 12:32 PM."


weather_tool = {
    "name": "get_weather",
    "description": "Gets the weather for in a given location",
    "input_schema": {
        "type": "object",
        "properties": {
            "location": {
                "type": "string",
                "description": "The city and state, e.g. San Francisco, CA",
            },
        },
        "required": ["location"],
    },
}

time_tool = {
    "name": "get_time",
    "description": "Gets the time in a given location",
    "input_schema": {
        "type": "object",
        "properties": {
            "location": {
                "type": "string",
                "description": "The city and state, e.g. San Francisco, CA",
            },
        },
        "required": ["location"],
    },
}


def process_tool_call(tool_name, tool_input):
    if tool_name == "get_weather":
        return get_weather(tool_input["location"])
    elif tool_name == "get_time":
        return get_time(tool_input["location"])
    else:
        raise ValueError(f"Unexpected tool name: {tool_name}")

Next, let's provide Claude with these tools and perform a query.

In [3]:
def make_query_and_print_result(messages, tools=None):
    response = client.messages.create(
        model=MODEL_NAME,
        messages=messages,
        max_tokens=1000,
        tool_choice={"type": "auto"},
        tools=tools or [weather_tool, time_tool],
    )

    for block in response.content:
        match block.type:
            case "text":
                print(block.text)
            case "tool_use":
                print(f"Tool: {block.name}({block.input})")
            case "thinking" | "redacted_thinking":
                pass  # the model may emit a leading thinking block; nothing to print
            case _:
                raise ValueError(f"Unexpected block type: {block.type}")

    return response


MESSAGES = [{"role": "user", "content": "What's the weather and time in San Francisco?"}]

response = make_query_and_print_result(MESSAGES)

Tool: get_weather({'location': 'San Francisco, CA'})
Tool: get_time({'location': 'San Francisco, CA'})


In the recorded run Claude asked for both tools in the same turn, which is the common case on current models. On other prompts or models you may get a single call here and the second one only after you return the first result, costing an extra round trip.

Either way, the right client pattern is the same: answer **every** `tool_use` block in the turn with a matching `tool_result`, all in one user message, then continue.

In [4]:
# Answer every tool_use block in the turn (there may be one or several)
tool_calls = [block for block in response.content if block.type == "tool_use"]

MESSAGES.append({"role": "assistant", "content": response.content})
MESSAGES.append(
    {
        "role": "user",
        "content": [
            {
                "type": "tool_result",
                "tool_use_id": tool_call.id,
                "content": process_tool_call(tool_call.name, tool_call.input),
            }
            for tool_call in tool_calls
        ],
    }
)

response = make_query_and_print_result(MESSAGES)

In San Francisco, CA it's 12:32 PM. The weather is sunny and 72°F.


With both results returned together, Claude answers in a single follow-up. If your run produced the calls one at a time instead, you paid for an extra request/response cycle: that sequential "back and forth" is what the batch tool below is designed to discourage.

## Introducing a batch tool

Let's introduce a `batch_tool`, so that Claude can have an opportunity to use it to combine multiple tool calls into one.

In [5]:
import json

batch_tool = {
    "name": "batch_tool",
    "description": "Invoke multiple other tool calls simultaneously",
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "The tool calls to invoke",
                "items": {
                    "types": "object",
                    "properties": {
                        "name": {
                            "types": "string",
                            "description": "The name of the tool to invoke",
                        },
                        "arguments": {
                            "types": "string",
                            "description": "The arguments to the tool",
                        },
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
}


def process_tool_with_maybe_batch(tool_name, tool_input):
    if tool_name == "batch_tool":
        results = []
        for invocation in tool_input["invocations"]:
            results.append(
                process_tool_call(invocation["name"], json.loads(invocation["arguments"]))
            )
        return "\n".join(results)
    else:
        return process_tool_call(tool_name, tool_input)

Now let's try to provide Claude with the existing weather and time tool, along with this new batch tool, and see what happens when we make a query requiring the weather and time.

In [6]:
MESSAGES = [{"role": "user", "content": "What's the weather and time in San Francisco?"}]

response = make_query_and_print_result(MESSAGES, tools=[weather_tool, time_tool, batch_tool])

Tool: get_weather({'location': 'San Francisco, CA'})
Tool: get_time({'location': 'San Francisco, CA'})


With the batch tool available, Claude can either wrap the calls in a single `batch_tool` invocation or, as in the recorded run, simply call `get_weather` and `get_time` in parallel directly. Both give you every call in one turn. The helper below handles either shape, unpacking `batch_tool` invocations when they appear.

In [7]:
# Answer every tool_use block in the turn; batch_tool invocations are unpacked by the helper
tool_calls = [block for block in response.content if block.type == "tool_use"]

MESSAGES.append({"role": "assistant", "content": response.content})
MESSAGES.append(
    {
        "role": "user",
        "content": [
            {
                "type": "tool_result",
                "tool_use_id": tool_call.id,
                "content": process_tool_with_maybe_batch(tool_call.name, tool_call.input),
            }
            for tool_call in tool_calls
        ],
    }
)

response = make_query_and_print_result(MESSAGES)

In San Francisco, CA, it's 12:32 PM, and the weather is sunny and 72°F.
